# SQLite Manual Analysis

Use this notebook to inspect the SQLite database created by the Rekordbox ingestion pipeline.

In [ ]:
from pathlib import Path
import os
import sqlite3

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if load_dotenv:
    load_dotenv(PROJECT_ROOT / ".env")

db_path = Path(os.getenv("SQLITE_DB_PATH", "data/rekordbox_tracks.sqlite3"))
if not db_path.is_absolute():
    db_path = PROJECT_ROOT / db_path

print(f"Project root: {PROJECT_ROOT}")
print(f"SQLite database: {db_path}")
print(f"Database exists: {db_path.exists()}")

## List Tables

In [ ]:
if not db_path.exists():
    raise FileNotFoundError(f"SQLite database not found: {db_path}")

conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

tables = conn.execute(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """
).fetchall()

table_names = [row["name"] for row in tables]
print("Tables:")
for table_name in table_names:
    print(f"- {table_name}")

table_names

## Print Schemas And Row Counts

In [ ]:
for table_name in table_names:
    print(f"\n=== {table_name} ===")
    row_count = conn.execute(f'SELECT COUNT(*) AS count FROM "{table_name}"').fetchone()["count"]
    print(f"Rows: {row_count}")
    print("Columns:")
    for column in conn.execute(f'PRAGMA table_info("{table_name}")').fetchall():
        print(
            f"- {column['name']} {column['type']} "
            f"nullable={not bool(column['notnull'])} pk={bool(column['pk'])}"
        )

## Load Rekordbox Tracks

In [ ]:
try:
    import pandas as pd
except ImportError as exc:
    raise ImportError("Install pandas to use the analysis cells: pip install pandas") from exc

tracks = pd.read_sql_query("SELECT * FROM rekordbox_tracks", conn)
print(f"Loaded {len(tracks)} tracks")
tracks.head(20)

## Load Spotify Enrichment Tables

Load the normalized Spotify enrichment output tables when they exist. This keeps the notebook usable before and after running `scripts/enrich_spotify.py`.

In [ ]:
def table_exists(table_name):
    return table_name in table_names


def table_columns(table_name):
    if not table_exists(table_name):
        return set()
    return {
        row["name"]
        for row in conn.execute(f'PRAGMA table_info("{table_name}")').fetchall()
    }

if table_exists("spotify_tracks"):
    spotify_tracks = pd.read_sql_query("SELECT * FROM spotify_tracks", conn)
else:
    spotify_tracks = pd.DataFrame()

if table_exists("rekordbox_spotify_matches"):
    rekordbox_spotify_matches = pd.read_sql_query(
        "SELECT * FROM rekordbox_spotify_matches",
        conn,
    )
else:
    rekordbox_spotify_matches = pd.DataFrame()

print(f"Loaded {len(spotify_tracks)} Spotify tracks")
print(f"Loaded {len(rekordbox_spotify_matches)} Rekordbox/Spotify matches")

if not spotify_tracks.empty:
    display(spotify_tracks.head(20))

if not rekordbox_spotify_matches.empty:
    display(rekordbox_spotify_matches.head(20))

## Build Rekordbox Spotify Match Base

Create a joined analysis base with every Rekordbox track, its accepted Spotify match when present, and the derived match score.

In [ ]:
if table_exists("spotify_tracks") and table_exists("rekordbox_spotify_matches"):
    track_columns = table_columns("rekordbox_tracks")
    match_columns = table_columns("rekordbox_spotify_matches")
    search_query_select = (
        "m.spotify_search_query_string"
        if "spotify_search_query_string" in match_columns
        else "rb.spotify_search_query_string"
        if "spotify_search_query_string" in track_columns
        else "NULL AS spotify_search_query_string"
    )
    rekordbox_spotify_match_base = pd.read_sql_query(
        """
        SELECT
            rb.rekordbox_track_id,
            rb.title AS rekordbox_title,
            rb.artist AS rekordbox_artist,
            rb.album AS rekordbox_album,
            rb.genre,
            rb.bpm,
            rb.key,
            rb.duration AS rekordbox_duration_seconds,
            rb.playlist_name,
            m.spotify_track_id,
            m.match_score,
            m.matched_at,
            {search_query_select},
            st.title AS spotify_title,
            st.artist_names AS spotify_artist_names,
            st.album_name AS spotify_album_name,
            st.duration_ms AS spotify_duration_ms,
            st.explicit AS spotify_explicit,
            st.spotify_url,
            CASE
                WHEN m.spotify_track_id IS NULL THEN 'unmatched'
                ELSE 'matched'
            END AS spotify_match_status
        FROM rekordbox_tracks rb
        LEFT JOIN rekordbox_spotify_matches m
            ON rb.rekordbox_track_id = m.rekordbox_track_id
        LEFT JOIN spotify_tracks st
            ON m.spotify_track_id = st.spotify_track_id
        ORDER BY rb.rekordbox_track_id
        """.format(search_query_select=search_query_select),
        conn,
    )
else:
    rekordbox_spotify_match_base = tracks.copy()
    if "spotify_search_query_string" not in rekordbox_spotify_match_base.columns:
        rekordbox_spotify_match_base["spotify_search_query_string"] = None
    rekordbox_spotify_match_base["spotify_match_status"] = "spotify tables missing"

print(f"Loaded {len(rekordbox_spotify_match_base)} rows in match base")
rekordbox_spotify_match_base.head(20)

## Quick Manual Analysis

In [ ]:
summary = {
    "tracks": len(tracks),
    "artists": tracks["artist"].nunique(dropna=True) if "artist" in tracks else None,
    "genres": tracks["genre"].nunique(dropna=True) if "genre" in tracks else None,
    "playlists": tracks["playlist_name"].nunique(dropna=True) if "playlist_name" in tracks else None,
    "avg_bpm": tracks["bpm"].mean() if "bpm" in tracks else None,
    "spotify_tracks": len(spotify_tracks),
    "spotify_matches": len(rekordbox_spotify_matches),
    "spotify_match_rate": (
        len(rekordbox_spotify_matches) / len(tracks)
        if len(tracks) else None
    ),
}
summary

## Spotify Match Analysis

In [ ]:
if "spotify_match_status" in rekordbox_spotify_match_base:
    display(
        rekordbox_spotify_match_base["spotify_match_status"]
        .value_counts(dropna=False)
        .rename_axis("spotify_match_status")
        .reset_index(name="tracks")
    )

if "match_score" in rekordbox_spotify_match_base and rekordbox_spotify_match_base["match_score"].notna().any():
    display(rekordbox_spotify_match_base["match_score"].describe())
    display(
        rekordbox_spotify_match_base
        .dropna(subset=["match_score"])
        .sort_values("match_score")
        [[
            "rekordbox_track_id",
            "rekordbox_title",
            "rekordbox_artist",
            "spotify_track_id",
            "spotify_title",
            "spotify_artist_names",
            "match_score",
            "spotify_search_query_string",
            "spotify_url",
        ]]
        .head(20)
    )
else:
    print("No Spotify match scores available yet. Run scripts/enrich_spotify.py to populate matches.")

In [ ]:
if "playlist_name" in tracks:
    display(tracks["playlist_name"].value_counts().rename_axis("playlist_name").reset_index(name="tracks"))

if "genre" in tracks:
    display(tracks["genre"].value_counts(dropna=False).head(20).rename_axis("genre").reset_index(name="tracks"))

if "artist" in tracks:
    display(tracks["artist"].value_counts(dropna=False).head(20).rename_axis("artist").reset_index(name="tracks"))

In [ ]:
unmatched_columns = [
    "rekordbox_track_id",
    "rekordbox_title",
    "rekordbox_artist",
    "rekordbox_album",
    "spotify_search_query_string",
    "spotify_match_status",
]
rekordbox_spotify_match_base.loc[
    rekordbox_spotify_match_base["spotify_match_status"] == "unmatched",
    [column for column in unmatched_columns if column in rekordbox_spotify_match_base.columns],
]